╔══════════════════════════════════════════════════════════════════════════════╗
║         CROP YIELD CLASSIFIER — Production-Ready Pipeline                   ║
║         Algorithm : XGBoost (GPU) + LightGBM (GPU) + Optuna Tuning         ║
║         Target    : Yield_tons_per_hectare → Low / Medium / High            ║
║         Hardware  : Google Colab H100 GPU                                   ║
╚══════════════════════════════════════════════════════════════════════════════╝

HOW TO USE IN COLAB:
  1. Upload crop_yield.csv to /content/
  2. Upload this file to /content/
  3. Run:  !python crop_yield_classifier.py
     OR copy each section into separate Colab cells.

OUTPUTS (saved to /content/model_artifacts/):
  - best_model.ubj          → Final XGBoost model (binary)
  - lgbm_model.txt          → LightGBM model
  - preprocessor.pkl        → sklearn pipeline (for inference)
  - label_encoder.pkl       → target LabelEncoder
  - feature_names.json      → ordered feature list
  - class_thresholds.json   → yield bin thresholds
  - classification_report.txt
  - confusion_matrix.png
  - feature_importance.png


──────────────────────────────────────────────────────────────────────────────
SECTION 0 │ Install dependencies  (run once)
──────────────────────────────────────────────────────────────────────────────


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("xgboost>=2.0")
install("lightgbm>=4.3")
install("optuna>=3.6")
install("scikit-learn>=1.4")
install("pandas>=2.0")
install("numpy>=1.26")
install("matplotlib>=3.8")
install("seaborn>=0.13")
install("joblib")
install("tqdm")

print("✅ All packages installed.")




──────────────────────────────────────────────────────────────────────────────
SECTION 1 │ Imports
──────────────────────────────────────────────────────────────────────────────


In [ ]:
import os, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from optuna.samplers import TPESampler
from tqdm import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import (LabelEncoder, OrdinalEncoder,
                                   StandardScaler, label_binarize)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score, f1_score,
                             ConfusionMatrixDisplay)

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Output directory
ARTIFACT_DIR = "/content/model_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("✅ Imports done.")
print(f"   XGBoost  : {xgb.__version__}")
print(f"   LightGBM : {lgb.__version__}")
print(f"   Optuna   : {optuna.__version__}")




──────────────────────────────────────────────────────────────────────────────
SECTION 2 │ Load Data
──────────────────────────────────────────────────────────────────────────────


In [ ]:
DATA_PATH = "/content/crop_yield.csv"

print(f"\n📂 Loading {DATA_PATH} …")
t0 = time.time()
df = pd.read_csv(DATA_PATH)
print(f"   Shape : {df.shape}  ({time.time()-t0:.1f}s)")
print(df.head(3))
print("\n📊 Data types:\n", df.dtypes)
print("\n📊 Nulls:\n", df.isnull().sum())




──────────────────────────────────────────────────────────────────────────────
SECTION 3 │ Target Engineering  (Regression → Classification)
──────────────────────────────────────────────────────────────────────────────


In [ ]:
TARGET_RAW = "Yield_tons_per_hectare"

# Quantile-based binning → balanced classes
q33 = df[TARGET_RAW].quantile(0.333)
q66 = df[TARGET_RAW].quantile(0.666)

thresholds = {"low_upper": round(q33, 4), "high_lower": round(q66, 4)}
print(f"\n🎯 Yield bins  →  Low < {q33:.3f} ≤ Medium < {q66:.3f} ≤ High")

def yield_to_class(y):
    if y < q33:
        return "Low"
    elif y < q66:
        return "Medium"
    else:
        return "High"

df["Yield_Class"] = df[TARGET_RAW].apply(yield_to_class)
print("\n📊 Class distribution:")
print(df["Yield_Class"].value_counts())

# Save thresholds
with open(f"{ARTIFACT_DIR}/class_thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)




──────────────────────────────────────────────────────────────────────────────
SECTION 4 │ Feature Engineering
──────────────────────────────────────────────────────────────────────────────


In [ ]:
print("\n🔧 Engineering features …")

# Boolean → int
bool_cols = ["Fertilizer_Used", "Irrigation_Used"]
for c in bool_cols:
    # handle both Python bool and string 'True'/'False'
    df[c] = df[c].map(
        {True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0}
    ).fillna(df[c].astype(str).str.lower().map({"true": 1, "false": 0}))
    df[c] = df[c].astype(int)

# Interaction features
df["Rain_Temp_Ratio"]   = df["Rainfall_mm"] / (df["Temperature_Celsius"] + 1e-3)
df["Fert_x_Irrig"]      = df["Fertilizer_Used"] * df["Irrigation_Used"]
df["Rain_per_Day"]      = df["Rainfall_mm"] / df["Days_to_Harvest"]
df["Temp_Days"]         = df["Temperature_Celsius"] * df["Days_to_Harvest"]
df["Rain_sq"]           = df["Rainfall_mm"] ** 2
df["Temp_sq"]           = df["Temperature_Celsius"] ** 2
df["Days_sq"]           = df["Days_to_Harvest"] ** 2
df["Fert_Rain"]         = df["Fertilizer_Used"]  * df["Rainfall_mm"]
df["Irrig_Temp"]        = df["Irrigation_Used"]  * df["Temperature_Celsius"]
df["Fert_Temp"]         = df["Fertilizer_Used"]  * df["Temperature_Celsius"]

print("   Features after engineering:", df.shape[1])




──────────────────────────────────────────────────────────────────────────────
SECTION 5 │ Define Feature Sets & Encode Target
──────────────────────────────────────────────────────────────────────────────


In [ ]:
CAT_COLS = ["Region", "Soil_Type", "Crop", "Weather_Condition"]
NUM_COLS = [
    "Rainfall_mm", "Temperature_Celsius", "Days_to_Harvest",
    "Fertilizer_Used", "Irrigation_Used",
    "Rain_Temp_Ratio", "Fert_x_Irrig", "Rain_per_Day",
    "Temp_Days", "Rain_sq", "Temp_sq", "Days_sq",
    "Fert_Rain", "Irrig_Temp", "Fert_Temp"
]
FEATURE_COLS = CAT_COLS + NUM_COLS
TARGET = "Yield_Class"

# Label-encode target  Low→0, Medium→1, High→2  (alphabetical default)
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
X = df[FEATURE_COLS].copy()

print(f"\n🏷  Classes: {list(le.classes_)}  →  {list(range(len(le.classes_)))}")

# Save artifacts
joblib.dump(le, f"{ARTIFACT_DIR}/label_encoder.pkl")
with open(f"{ARTIFACT_DIR}/feature_names.json", "w") as f:
    json.dump({"categorical": CAT_COLS,
               "numerical":   NUM_COLS,
               "all_features": FEATURE_COLS,
               "classes":     list(le.classes_)}, f, indent=2)




──────────────────────────────────────────────────────────────────────────────
SECTION 6 │ Preprocessing Pipeline
──────────────────────────────────────────────────────────────────────────────
OrdinalEncoder for tree models (no one-hot needed; XGB/LGBM handle ordinals)


In [ ]:
ord_enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=np.float32
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ord_enc,     CAT_COLS),
        ("num", "passthrough", NUM_COLS),   # trees don't need scaling
    ],
    remainder="drop"
)



Save preprocessor (will be fit on train set below)
(saved after fitting in Section 7)


In [ ]:

print("✅ Preprocessor defined.")




──────────────────────────────────────────────────────────────────────────────
SECTION 7 │ Train / Validation / Test Split
──────────────────────────────────────────────────────────────────────────────


In [ ]:
print("\n✂️  Splitting data  (70 / 15 / 15) …")

X_train_raw, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

# Fit preprocessor only on train
X_train = preprocessor.fit_transform(X_train_raw).astype(np.float32)
X_val   = preprocessor.transform(X_val_raw).astype(np.float32)
X_test  = preprocessor.transform(X_test_raw).astype(np.float32)

joblib.dump(preprocessor, f"{ARTIFACT_DIR}/preprocessor.pkl")

print(f"   Train : {X_train.shape}")
print(f"   Val   : {X_val.shape}")
print(f"   Test  : {X_test.shape}")

# XGBoost DMatrix (GPU-ready)
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURE_COLS)
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=FEATURE_COLS)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURE_COLS)

# LightGBM Dataset
lgb_train = lgb.Dataset(X_train, label=y_train, feature_name=FEATURE_COLS, free_raw_data=False)
lgb_val   = lgb.Dataset(X_val,   label=y_val,   feature_name=FEATURE_COLS, reference=lgb_train, free_raw_data=False)




──────────────────────────────────────────────────────────────────────────────
SECTION 8 │ Optuna Hyperparameter Tuning — XGBoost (GPU)
──────────────────────────────────────────────────────────────────────────────


In [ ]:
N_TRIALS   = 40      # increase to 80-100 for even better results
NUM_CLASSES = len(le.classes_)

print(f"\n🔍 Optuna XGBoost tuning  ({N_TRIALS} trials) …")

def xgb_objective(trial):
    params = {
        "device":           "cuda",
        "tree_method":      "hist",
        "objective":        "multi:softmax",
        "num_class":        NUM_CLASSES,
        "eval_metric":      "mlogloss",
        "verbosity":        0,
        "seed":             SEED,
        # tunable
        "n_estimators":     trial.suggest_int("n_estimators", 300, 1500),
        "max_depth":        trial.suggest_int("max_depth", 4, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bylevel":trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "max_bin":          trial.suggest_categorical("max_bin", [256, 512]),
    }
    n_est = params.pop("n_estimators")
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=n_est,
        evals=[(dval, "val")],
        early_stopping_rounds=30,
        verbose_eval=False,
    )
    preds = model.predict(dval).astype(int)
    return f1_score(y_val, preds, average="weighted")

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED)
)
study_xgb.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
print(f"\n   Best XGBoost F1 (val) : {study_xgb.best_value:.4f}")
print("   Best params:", best_xgb_params)




──────────────────────────────────────────────────────────────────────────────
SECTION 9 │ Train Final XGBoost Model
──────────────────────────────────────────────────────────────────────────────


In [ ]:
print("\n🚀 Training final XGBoost model …")

n_est_final = best_xgb_params.pop("n_estimators")

final_xgb_params = {
    "device":           "cuda",
    "tree_method":      "hist",
    "objective":        "multi:softprob",    # probabilities for AUC
    "num_class":        NUM_CLASSES,
    "eval_metric":      ["mlogloss", "merror"],
    "verbosity":        1,
    "seed":             SEED,
    **best_xgb_params,
}

xgb_model = xgb.train(
    final_xgb_params,
    dtrain,
    num_boost_round=n_est_final,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=30,
    verbose_eval=50,
)

xgb_model.save_model(f"{ARTIFACT_DIR}/best_model.ubj")
print("✅ XGBoost model saved.")




──────────────────────────────────────────────────────────────────────────────
SECTION 10 │ Optuna Hyperparameter Tuning — LightGBM (GPU)
──────────────────────────────────────────────────────────────────────────────


In [ ]:
print(f"\n🔍 Optuna LightGBM tuning  ({N_TRIALS} trials) …")

def lgb_objective(trial):
    params = {
        "device":           "gpu",
        "objective":        "multiclass",
        "num_class":        NUM_CLASSES,
        "metric":           "multi_logloss",
        "verbosity":        -1,
        "seed":             SEED,
        "n_estimators":     trial.suggest_int("n_estimators", 300, 1500),
        "num_leaves":       trial.suggest_int("num_leaves", 31, 255),
        "max_depth":        trial.suggest_int("max_depth", 4, 12),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_samples":trial.suggest_int("min_child_samples", 10, 100),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    n_est = params.pop("n_estimators")
    callbacks = [lgb.early_stopping(30, verbose=False),
                 lgb.log_evaluation(period=-1)]
    model = lgb.train(
        params, lgb_train,
        num_boost_round=n_est,
        valid_sets=[lgb_val],
        callbacks=callbacks,
    )
    preds = np.argmax(model.predict(X_val), axis=1)
    return f1_score(y_val, preds, average="weighted")

study_lgb = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED)
)
study_lgb.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_lgb_params = study_lgb.best_params
print(f"\n   Best LightGBM F1 (val) : {study_lgb.best_value:.4f}")




──────────────────────────────────────────────────────────────────────────────
SECTION 11 │ Train Final LightGBM Model
──────────────────────────────────────────────────────────────────────────────


In [ ]:
print("\n🚀 Training final LightGBM model …")

n_est_lgb = best_lgb_params.pop("n_estimators")
final_lgb_params = {
    "device":           "gpu",
    "objective":        "multiclass",
    "num_class":        NUM_CLASSES,
    "metric":           ["multi_logloss", "multi_error"],
    "verbosity":        -1,
    "seed":             SEED,
    **best_lgb_params,
}

callbacks_final = [lgb.early_stopping(30, verbose=False),
                   lgb.log_evaluation(period=50)]

lgb_model = lgb.train(
    final_lgb_params, lgb_train,
    num_boost_round=n_est_lgb,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=callbacks_final,
)

lgb_model.save_model(f"{ARTIFACT_DIR}/lgbm_model.txt")
print("✅ LightGBM model saved.")




──────────────────────────────────────────────────────────────────────────────
SECTION 12 │ Evaluation on Test Set
──────────────────────────────────────────────────────────────────────────────


In [ ]:
print("\n📊 Evaluating on held-out TEST set …")
CLASS_NAMES = list(le.classes_)   # ['High', 'Low', 'Medium']

# ── XGBoost ──
xgb_proba = xgb_model.predict(dtest).reshape(-1, NUM_CLASSES)
xgb_pred  = np.argmax(xgb_proba, axis=1)

xgb_acc  = accuracy_score(y_test, xgb_pred)
xgb_f1   = f1_score(y_test, xgb_pred, average="weighted")
xgb_auc  = roc_auc_score(
    label_binarize(y_test, classes=range(NUM_CLASSES)),
    xgb_proba, multi_class="ovr", average="weighted"
)
xgb_report = classification_report(y_test, xgb_pred, target_names=CLASS_NAMES)

# ── LightGBM ──
lgb_proba = lgb_model.predict(X_test)
lgb_pred  = np.argmax(lgb_proba, axis=1)

lgb_acc  = accuracy_score(y_test, lgb_pred)
lgb_f1   = f1_score(y_test, lgb_pred, average="weighted")
lgb_auc  = roc_auc_score(
    label_binarize(y_test, classes=range(NUM_CLASSES)),
    lgb_proba, multi_class="ovr", average="weighted"
)
lgb_report = classification_report(y_test, lgb_pred, target_names=CLASS_NAMES)

print("\n" + "═"*60)
print(f"{'MODEL':<20}  {'Accuracy':>10}  {'F1-Weighted':>12}  {'ROC-AUC':>10}")
print("─"*60)
print(f"{'XGBoost (GPU)':<20}  {xgb_acc:>10.4f}  {xgb_f1:>12.4f}  {xgb_auc:>10.4f}")
print(f"{'LightGBM (GPU)':<20}  {lgb_acc:>10.4f}  {lgb_f1:>12.4f}  {lgb_auc:>10.4f}")
print("═"*60)
print("\nXGBoost Classification Report:\n", xgb_report)
print("\nLightGBM Classification Report:\n", lgb_report)

# Save text report
with open(f"{ARTIFACT_DIR}/classification_report.txt", "w") as f:
    f.write("XGBoost\n" + "="*60 + "\n")
    f.write(f"Accuracy : {xgb_acc:.4f}\n")
    f.write(f"F1-Weighted : {xgb_f1:.4f}\n")
    f.write(f"ROC-AUC : {xgb_auc:.4f}\n\n")
    f.write(xgb_report)
    f.write("\n\nLightGBM\n" + "="*60 + "\n")
    f.write(f"Accuracy : {lgb_acc:.4f}\n")
    f.write(f"F1-Weighted : {lgb_f1:.4f}\n")
    f.write(f"ROC-AUC : {lgb_auc:.4f}\n\n")
    f.write(lgb_report)




──────────────────────────────────────────────────────────────────────────────
SECTION 13 │ Plots
──────────────────────────────────────────────────────────────────────────────


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrices
for ax, pred, name in zip(axes,
                           [xgb_pred, lgb_pred],
                           ["XGBoost (GPU)", "LightGBM (GPU)"]):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}", fontsize=13, fontweight="bold")

plt.suptitle("Confusion Matrices — Test Set", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrix saved.")

# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# XGBoost
xgb_imp = pd.Series(xgb_model.get_score(importance_type="gain"),
                     name="Gain").sort_values(ascending=False).head(20)
xgb_imp.plot(kind="barh", ax=axes[0], color="#2196F3", edgecolor="white")
axes[0].set_title("XGBoost — Top 20 Features (Gain)", fontweight="bold")
axes[0].invert_yaxis()

# LightGBM
lgb_imp = pd.Series(lgb_model.feature_importance(importance_type="gain"),
                     index=FEATURE_COLS,
                     name="Gain").sort_values(ascending=False).head(20)
lgb_imp.plot(kind="barh", ax=axes[1], color="#4CAF50", edgecolor="white")
axes[1].set_title("LightGBM — Top 20 Features (Gain)", fontweight="bold")
axes[1].invert_yaxis()

plt.suptitle("Feature Importance", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Feature importance plot saved.")




──────────────────────────────────────────────────────────────────────────────
SECTION 14 │ Production Inference Function
──────────────────────────────────────────────────────────────────────────────


In [ ]:
def load_pipeline(artifact_dir: str = ARTIFACT_DIR):


Load all saved artifacts for inference.


In [ ]:
    """Load all saved artifacts for inference."""
    model.load_model(f"{artifact_dir}/best_model.ubj")
    pre       = joblib.load(f"{artifact_dir}/preprocessor.pkl")
    enc       = joblib.load(f"{artifact_dir}/label_encoder.pkl")
    with open(f"{artifact_dir}/feature_names.json") as f:
        feat_info = json.load(f)
    with open(f"{artifact_dir}/class_thresholds.json") as f:
        thresholds = json.load(f)
    return model, pre, enc, feat_info, thresholds


def predict(raw_records: list[dict],
            artifact_dir: str = ARTIFACT_DIR) -> pd.DataFrame:


    Predict crop yield class from raw input records.

    Parameters
    ----------
    raw_records : list of dicts, each with keys:
        Region, Soil_Type, Crop, Rainfall_mm, Temperature_Celsius,
        Fertilizer_Used, Irrigation_Used, Weather_Condition, Days_to_Harvest

    Returns
    -------
    pd.DataFrame with columns: Predicted_Class, Probability_High,
                                Probability_Low, Probability_Medium
    


In [ ]:
    model, pre, enc, feat_info, _ = load_pipeline(artifact_dir)

    df_input = pd.DataFrame(raw_records)

    # Bool → int
    for c in ["Fertilizer_Used", "Irrigation_Used"]:
        df_input[c] = df_input[c].map(
            {True: 1, False: 0, "True": 1, "False": 0,
             "true": 1, "false": 0, 1: 1, 0: 0}
        ).astype(int)

    # Feature engineering (same as training)
    df_input["Rain_Temp_Ratio"]   = df_input["Rainfall_mm"] / (df_input["Temperature_Celsius"] + 1e-3)
    df_input["Fert_x_Irrig"]      = df_input["Fertilizer_Used"] * df_input["Irrigation_Used"]
    df_input["Rain_per_Day"]      = df_input["Rainfall_mm"] / df_input["Days_to_Harvest"]
    df_input["Temp_Days"]         = df_input["Temperature_Celsius"] * df_input["Days_to_Harvest"]
    df_input["Rain_sq"]           = df_input["Rainfall_mm"] ** 2
    df_input["Temp_sq"]           = df_input["Temperature_Celsius"] ** 2
    df_input["Days_sq"]           = df_input["Days_to_Harvest"] ** 2
    df_input["Fert_Rain"]         = df_input["Fertilizer_Used"] * df_input["Rainfall_mm"]
    df_input["Irrig_Temp"]        = df_input["Irrigation_Used"] * df_input["Temperature_Celsius"]
    df_input["Fert_Temp"]         = df_input["Fertilizer_Used"] * df_input["Temperature_Celsius"]

    X_inf = pre.transform(df_input[feat_info["all_features"]]).astype(np.float32)
    dm    = xgb.DMatrix(X_inf, feature_names=feat_info["all_features"])
    proba = model.predict(dm).reshape(-1, len(enc.classes_))

    pred_idx    = np.argmax(proba, axis=1)
    pred_labels = enc.inverse_transform(pred_idx)

    result = pd.DataFrame(proba, columns=[f"Prob_{c}" for c in enc.classes_])
    result.insert(0, "Predicted_Class", pred_labels)
    return result


# ── Quick inference demo ──
print("\n🧪 Inference demo (3 sample rows):")
sample_records = [
    {
        "Region": "North", "Soil_Type": "Loam", "Crop": "Wheat",
        "Rainfall_mm": 824.4, "Temperature_Celsius": 29.8,
        "Fertilizer_Used": False, "Irrigation_Used": False,
        "Weather_Condition": "Sunny", "Days_to_Harvest": 100
    },
    {
        "Region": "South", "Soil_Type": "Clay", "Crop": "Rice",
        "Rainfall_mm": 992.7, "Temperature_Celsius": 18.0,
        "Fertilizer_Used": True, "Irrigation_Used": True,
        "Weather_Condition": "Rainy", "Days_to_Harvest": 140
    },
    {
        "Region": "West", "Soil_Type": "Sandy", "Crop": "Cotton",
        "Rainfall_mm": 145.3, "Temperature_Celsius": 19.8,
        "Fertilizer_Used": True, "Irrigation_Used": True,
        "Weather_Condition": "Cloudy", "Days_to_Harvest": 141
    },
]

demo_out = predict(sample_records)
print(demo_out.to_string(index=False))




──────────────────────────────────────────────────────────────────────────────
SECTION 15 │ Summary
──────────────────────────────────────────────────────────────────────────────


In [ ]:
print("\n" + "═"*60)
print("🏁  TRAINING COMPLETE")
print("═"*60)
print(f"   Artifacts saved to : {ARTIFACT_DIR}")
print(f"   XGBoost  →  Acc={xgb_acc:.4f}  F1={xgb_f1:.4f}  AUC={xgb_auc:.4f}")
print(f"   LightGBM →  Acc={lgb_acc:.4f}  F1={lgb_f1:.4f}  AUC={lgb_auc:.4f}")
print("\n   Files:")
for f in sorted(os.listdir(ARTIFACT_DIR)):
    size = os.path.getsize(f"{ARTIFACT_DIR}/{f}") / 1024
    print(f"     {f:<35} {size:>8.1f} KB")
print("═"*60)
